<span style="color: #6a737d; font-family: monospace;">
Created on Sat May 03 2025 21:59:02<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

from itertools import islice

import torch
from torch.utils.data import DataLoader

from usf.dataset.stanford2D3DS import Stanford2D3DSDataset
from usf.utils.files import read_file
from usf.visualization.spherical_projection import visualize_spherical_image

In [ ]:
stanford2d3ds_dataset_base_path = "data/stanford2D3DS"
augmentation = {
    "chroma_jitter": 0.5,
    "luma_jitter": 0.5,
    "gaussian_blur": 0.5,
    "gray_scale": 0.2,
    "horizontal_reflection": 0.5,
    "vertical_reflection": 0.5,
    "erase": 0.5,
    "rotation": 0.5,
}
meta = {
    "input": {
        "mean": [0.53989742, 0.51861451, 0.45900312],  # in [0, 1] scale
        "std": [0.19439127, 0.19148353, 0.19884944],  # in [0, 1] scale
        "range": [0.0, 255.0],
    },
    "label": {
        "ignore_index": 0,  # label index to ignore during training and validation
        "class_weight": [
            0.0,
            0.84401656,
            0.98537693,
            0.68518653,
            0.61882607,
            0.68802938,
            0.62075897,
            0.82498247,
            0.62132053,
            0.61883815,
            3.86012239,
            0.8564899,
            0.61882153,
            1.15104239,
        ],  # class weight for cross-entropy loss, computed offline, see compute_class_weights in Stanford2D3DSDataset}
    },
}
seed = "Semantic Segmentation"
downsample_image_size = (960, 480)
output_vector = read_file("config/lens_normal_map/180_180_560_560.npy")
output_vector_mask = read_file("config/lens_normal_map/180_180_560_560_mask.npy")
batch_size = 4

In [ ]:
device = "cuda"
dtype = torch.float32
factory_kwargs = {"device": device, "dtype": dtype}

In [ ]:
stanford2d3ds_train_dataset = Stanford2D3DSDataset(
    dataset_base_path=stanford2d3ds_dataset_base_path,
    dataset_type="train",
    augmentation=augmentation,
    downsample_image_size=downsample_image_size,
    output_vector=output_vector,
    output_vector_mask=output_vector_mask,
    meta=meta,
    seed=seed,
)
stanford2d3ds_train_dataloader = DataLoader(
    stanford2d3ds_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=stanford2d3ds_train_dataset.collate_fn,
)

In [ ]:
# Retrieve one batch from the DataLoader
batch_idx = 0
batch = stanford2d3ds_train_dataset.move_batch_to(
    next(islice(stanford2d3ds_train_dataloader, batch_idx, None)),
    device=device,
    dtype=dtype,
)

In [ ]:
visualize_spherical_image(batch["inputs"]["spherical_images"], fps=1)

In [ ]:
figures, spherical_vis = stanford2d3ds_train_dataset.visualize_batch(batch)

In [ ]:
visualize_spherical_image(spherical_vis, fps=1)